[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gps/gps.ipynb)

# Satellite Navigation

Satellite navigation recovers a receiver's spatial position, clock offset, and spacetime uncertainty from the transmission timestamps of arriving signals across changing sky coverage. Each satellite transmission places the reception event on its future light cone, so four exact cones determine position and time together at their intersection, while noisy timestamps turn redundant cones into a coupled spacetime uncertainty region whose shape tracks how broadly the constellation spans the sky.

Projective spacetime geometric algebra models events as intrinsic antivectors and measurement slices as extrinsic vectors, with a null weight hyperplane carrying translations alongside the causal metric. The term ***duality map*** here means a linear map between a space and its dual, such as `Slice <- Event` or `Readout <- Step`. Each transmission event defines its light cone as the quadratic duality map `emitted | (emitted & Event)` whose pairing with an event vanishes along null rays, pairwise cone differences cancel the quadratic Lorentz term to yield radical hyperplanes whose wedge meets a cone at the reception event, and weighted tangent readouts sum into `(weight * readout * (readout & Step)).sum(axis=-1)`, a positive-definite information duality map whose `.inverse()` jointly captures spatial and clock uncertainty.

In pseudorange navigation and Bancroft matrix notation ([Bancroft's method](https://gssc.esa.int/navipedia/index.php/Bancroft_Method); [weighted least squares](https://gssc.esa.int/navipedia/index.php/Weighted_Least_Square_Solution_%28WLS%29); [dilution of precision](https://gssc.esa.int/navipedia/index.php?title=Positioning_Error)), each satellite writes a nonlinear Euclidean range-plus-clock-bias equation $\rho_i = \|\mathbf{r} - \mathbf{r}_i\| + c\,\delta t$, Bancroft's closed-form solver introduces a separate $4 \times 4$ diagonal Lorentz matrix $M = \operatorname{diag}(1, 1, 1, -1)$ with manual inner products $\langle \mathbf{a}, \mathbf{b}\rangle_M = \mathbf{a}^\top M \mathbf{b}$ and scalar quadratic roots, and dilution-of-precision analysis assembles an $N \times 4$ unit line-of-sight Jacobian $H_{i,:} = \bigl(-(\mathbf{r}_i - \mathbf{r})^\top / \|\mathbf{r}_i - \mathbf{r}\|,\; 1\bigr)$ to invert $(H^\top W H)^{-1}$ into spatial and clock covariance blocks.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, dim, stack, where
from examples.relativity.gps import render, scenarios
from examples.util.animation import save_figure, show_states

ga = Algebra("x+y+z+t-w0")
mv = NumpyContext(ga, units=True).multivector
Scalar = ga.gatype.scalar()                                                       # Scalar+
Event = ga.gatype.antivector()                                                    # Antivector-
Slice = ga.gatype.vector()                                                        # Vector+
Readout = Slice.derive.nondegenerate                                              # Vector+
Step = Readout.derive.dual                                                        # Antivector-
Cone = ga.gatype((Slice, Event))                                                  # Slice+ <- Event-
Information = ga.gatype((Readout, Step))                                          # Readout+ <- Step-
Covariance = ga.gatype((Step, Readout))                                           # Step- <- Readout+
Orbit = ga.gatype.bivector()                                                      # Bivector+
# Purely spatial readout planes spanning `x`, `y`, and `z`.
Spatial = Slice.derive.within(Algebra("x+y+z+"))                                  # Vector+

c = mv.scalar(1.0, unit=dim.length / dim.time)                                    # [] Scalar+, length/time
origin = mv.w.dual()                                                              # [] Event-
across = mv.x.dual()                                                              # [] Event-
time = -mv.t.dual()                                                               # [] Event-

## Retarded satellite transmissions

Orbiting satellites transmit from retarded positions along their circular polar trajectories so that all signals arrive at the polar receiver at coordinate-time zero. Measuring lengths in Earth radii and times in Earth-radius light-travel times sets the speed of light to unity, and fixed-point updates on the rest-frame spatial distance rapidly converge to the simultaneous-arrival transmission events.

In [ ]:
from examples.relativity.gps.scenarios import (
    angular_speed,
    earth_radius,
    earth_radius_metres,
    initial_y,
    initial_z,
    light_speed,
    n_emission_steps,
    n_satellites,
    orbit_radius,
    phase_offsets,
    snapshot_phase,
)

# Polar orbital planes share the `x` axis at evenly spaced longitudes and distinct orbital phases.
orbit_rotors = (mv.yz * (-np.arange(n_satellites) * (2 * np.pi / n_satellites) / 2)).exp()  # [n_satellites] Rotor+
orbital_planes = orbit_rotors >> mv.xy                                            # [n_satellites] Bivector+
phases = phase_offsets + snapshot_phase

# Each snapshot places the reception epoch at coordinate-time zero above the pole.
receiver = origin + across * earth_radius                                         # [] Event-
coarse_position = receiver + mv.y.dual() * initial_y + mv.z.dual() * initial_z    # [] Event-

In [ ]:
def orbit_positions(planes: Orbit, phases: np.ndarray, radius: Scalar,
                    angular_speed: Scalar, times: Scalar) -> Event:
    """Circular satellite trajectories in `planes` (`[n_satellites] Bivector+`) at `phases`, `radius` (`length`), and
    `angular_speed` (`1/time`) take coordinate `times` (`[...] Scalar+`, `time`) to `[..., n_satellites] Event-`."""
    rotations = (-planes * ((phases + angular_speed * times) / 2)).exp()          # [..., n_satellites] Rotor+
    return origin + (rotations >> across) * radius + time * (c * times)           # [..., n_satellites] Event-


def spatial_distance(first: Event, second: Event) -> Scalar:
    """Rest-frame spatial distance (`[...] Scalar+`) between unit-weight events `first` and `second`
    combines the squared coordinate-time separation with minus the spacetime join's squared norm."""
    return ((mv.t & (first - second)).squared() - (first & second).scalar_norm_squared()).square_root()

In [ ]:
def emitted_events(receiver: Event, planes: Orbit, phases: np.ndarray, radius: Scalar,
                   angular_speed: Scalar, n_steps: int) -> Event:
    """Fixed-point iteration over `n_steps` light-time updates locates the retarded satellite
    transmission events (`[n_satellites] Event-`) whose signals arrive together at `receiver`."""
    reception_time = (mv.t & receiver) / c                                        # [] Scalar+, time
    emission_times = reception_time                                               # [] Scalar+, time
    for _ in range(n_steps):
        satellites = orbit_positions(planes, phases, radius, angular_speed, emission_times)  # [n_satellites] Event-
        emission_times = reception_time - spatial_distance(receiver, satellites) / c  # [n_satellites] Scalar+, time
    return orbit_positions(planes, phases, radius, angular_speed, emission_times)  # [n_satellites] Event-


emitted = emitted_events(receiver, orbital_planes, phases, orbit_radius, angular_speed, n_emission_steps)
save_figure(render.constellation_figure(emitted, receiver, orbit_radius), "gps_constellation");

The orange marker on Earth's polar cap shows the receiver, the six orange-and-blue craft mark the satellites at their retarded transmission epochs, and the blue rays trace the light paths converging on reception.

## Exact four-cone intersection

Contracting the spacetime join `emitted & Event` with the transmission vertex gives the light-cone duality map `emitted | (emitted & Event)`, whose hyperplane `cone(event)` is tangent to the cone whenever `cone(event) & event` vanishes. Subtracting one light cone from three others cancels the shared quadratic Lorentz part and leaves three radical hyperplanes whose wedge is a spacetime line; intersecting that line with one cone yields two null candidates, and causal ordering picks the unique event occurring after every transmission.

In [ ]:
def light_cones(emitted: Event) -> Cone:
    """Transmission vertices `emitted` (`[..., n_satellites] Event-`) define light-cone quadratic duality maps
    `emitted | (emitted & Event)` (`[..., n_satellites] Slice+ <- Event-`)."""
    return emitted | (emitted & Event)                                            # [..., n_satellites] Slice+ <- Event-


def cone_intersections(cones: Cone) -> Event:
    """Pairwise radical planes of four `cones` (`[4] Slice+ <- Event-`) meet in a spacetime line whose
    two intersections (`[2] Event-`) with `cones[0]` solve the four-satellite light-cone system."""
    # Subtracting two light cones cancels the quadratic Lorentz part, leaving the radical hyperplane paired with `w`.
    difference = cones[1:] - cones[0]                                             # [3] Slice+ <- Event-
    at_origin = difference(origin)                                                # [3] Slice+
    radicals = at_origin - mv.w * ((at_origin & origin) / 2)                      # [3] Slice+
    axis = radicals[0] ^ radicals[1] ^ radicals[2]                                # [] Trivector+

    # Meeting the radical line with `t = 0` and with the ideal hyperplane `w` gives a point on the line and its direction.
    base_cut = axis ^ mv.t
    base = base_cut / (mv.w & base_cut)                                           # [] Event-
    ideal = axis ^ mv.w
    direction = ideal / (mv.t & ideal)                                            # [] Event-

    # Solving the quadratic interval along `base + parameter * direction` locates the two null intersections.
    quadratic = cones[0](direction) & direction                                   # [] Scalar+
    linear = cones[0](direction) & base                                           # [] Scalar+
    constant = cones[0](base) & base                                              # [] Scalar+
    discriminant_root = (linear.squared() - quadratic * constant).square_root()   # [] Scalar+
    parameters = stack((-linear - discriminant_root, -linear + discriminant_root)) / quadratic  # [2] Scalar+
    return base + direction * parameters                                          # [2] Event-

In [ ]:
def causal_reception(candidates: Event, emitted: Event) -> Event:
    """Causal ordering selects from `candidates` (`[2] Event-`) the reception event (`[] Event-`)
    occurring after every transmission in `emitted` (`[n_signals] Event-`)."""
    after = (mv.t & (candidates[:, None] - emitted)) > 0                          # [2, n_signals]
    return where(after[0].all(), candidates[0], candidates[1])                    # [] Event-


cones = light_cones(emitted[:4])                                                  # [4] Slice+ <- Event-
candidates = cone_intersections(cones)                                            # [2] Event-
located = causal_reception(candidates, emitted[:4])                               # [] Event-
# Only the causal root sits near Earth's surface at reception epoch zero; the spurious root lies far in the past.
candidate_distances = spatial_distance(candidates, coarse_position) / dim.length * (earth_radius_metres / 1000)
candidate_times = ((mv.t & candidates) / c) / dim.time * (earth_radius_metres / light_speed * 1000)
print("distances from the coarse position (km):", candidate_distances.to_array())
print("reception times (ms):", candidate_times.to_array())

## Least-squares reception and spacetime covariance

Timestamp errors shift the measured cone vertices in time so that redundant light cones no longer share a common event. Gauss-Newton fitting drives each squared interval `cones(receiver) & receiver` toward zero, weighting every satellite by the inverse square of how far timestamp noise perturbs its interval (twice the light-travel time). Projecting out the null weight direction from `2 * cones(receiver)` gives the displacement readout `Readout`, whose weighted dyads sum into the positive-definite `Information` duality map; inverting that map yields the joint position-and-clock `Covariance`, and its spatial restriction furnishes the one-standard-deviation position ellipsoid.

In [ ]:
from examples.relativity.gps.scenarios import (
    clock_offset,
    n_position_steps,
    sigma,
    timestamp_noise,
)

# Timestamp noise shifts the measured cone vertices in time, and clock bias offsets the initial guess.
observed = emitted + time * (c * timestamp_noise)                                 # [n_satellites] Event-
initial = coarse_position + time * (c * clock_offset)                             # [] Event-

In [ ]:
def cone_intervals(cones: Cone, emitted: Event, receiver: Event) -> tuple[Scalar, Readout, Scalar]:
    """Pairing `cones` (`[..., n_satellites] Slice+ <- Event-`) with `receiver` yields the squared
    intervals (`[..., n_satellites] Scalar+`), displacement readouts (`[..., n_satellites] Readout+`),
    and timestamp error levers (`[..., n_satellites] Scalar+`)."""
    tangent = cones(receiver[..., None])                                          # [..., n_satellites] Slice+
    # Projecting out `w` retains the nondegenerate spacetime 1-form reading displacements `Step-`.
    readout = 2 * tangent.cast(Readout)                                           # [..., n_satellites] Readout+
    lever = 2 * (mv.t & (receiver[..., None] - emitted))                          # [..., n_satellites] Scalar+
    return tangent & receiver[..., None], readout, lever


def fitted_reception(emitted: Event, initial: Event, sigma: Scalar,
                     n_steps: int) -> tuple[Event, Information, Covariance]:
    """Gauss-Newton iteration over `n_steps` fits the reception event (`[...] Event-`) to noisy
    `emitted` cone vertices with range uncertainty `sigma`, along with `Information` and `Covariance`."""
    cones = light_cones(emitted)                                                  # [..., n_satellites] Slice+ <- Event-
    receiver = initial                                                            # [...] Event-
    for _ in range(n_steps):
        interval, readout, lever = cone_intervals(cones, emitted, receiver)
        weight = 1 / (lever * sigma) ** 2                                         # [..., n_satellites] Scalar+
        information = (weight * readout * (readout & Step)).sum(axis=-1)          # [...] Readout+ <- Step-
        mismatch = (weight * readout * interval).sum(axis=-1)                     # [...] Readout+
        receiver = receiver - information.solve(mismatch)                         # [...] Event-

    _, readout, lever = cone_intervals(cones, emitted, receiver)
    information = (readout * (readout & Step) / (lever * sigma) ** 2).sum(axis=-1)  # [...] Readout+ <- Step-
    covariance = information.inverse()                                            # [...] Step- <- Readout+
    return receiver, information, covariance

In [ ]:
def position_semi_axes(covariance: Covariance) -> Event:
    """Principal spatial eigendecomposition of `covariance` (`[...] Step- <- Readout+`) gives the
    one-standard-deviation position ellipsoid semi-axes (`[..., 3] Event-`)."""
    variances, principal = (Spatial & covariance(Spatial)).eigh()                 # [..., 3] Scalar+, Readout+
    return covariance(principal) / variances.square_root()                        # [..., 3] Event-


estimate, information, covariance = fitted_reception(observed, initial, sigma, n_position_steps)
semi_axes = position_semi_axes(covariance)                                        # [3] Event-
clock_offset_us = (clock_offset - (mv.t & estimate) / c) / dim.time * (earth_radius_metres / light_speed * 1e6)
clock_spread_ns = ((mv.t & covariance(mv.t)).square_root() / c) / dim.time * (earth_radius_metres / light_speed * 1e9)
position_error = spatial_distance(estimate, receiver) / dim.length * earth_radius_metres
position_spreads = spatial_distance(origin + semi_axes, origin) / dim.length * earth_radius_metres
print("recovered clock offset (µs):", clock_offset_us.to_array())
print("clock uncertainty (ns):", clock_spread_ns.to_array())
print("position error (m):", position_error.to_array())
print("position uncertainties along the principal axes (m):", position_spreads.to_array())
save_figure(render.positioning_figure(emitted, receiver, estimate, semi_axes, orbit_radius), "gps_positioning");

The orange sphere marks the true polar receiver, the blue sphere marks the Gauss-Newton estimate, and the translucent blue surface with three principal diameters shows the one-standard-deviation position ellipsoid. Exaggerating the range uncertainty to 1,500 km makes the ellipsoid visible at orbital scale, where its radial elongation reflects the coupling between vertical motion and receiver-clock bias.

## Sky coverage and dilution of precision

When satellites cluster overhead near the pole, radial receiver displacements shift all signal ranges nearly together and mimic a receiver clock offset, inflating vertical uncertainty. As the constellation fans out toward the horizon across orbital phases under a fixed timestamp-noise draw, the light-cone tangent hyperplanes separate in angle and the positive-definite information duality map tightens in every spatial direction.

In [ ]:
from examples.relativity.gps.scenarios import first_phase, frame_duration_ms, last_phase, n_frames


def sky_spread_states(n_steps: int) -> Iterator[tuple[Event, Event, Event, Event]]:
    """Yields `(emitted, receiver, estimate, semi_axes)` across `n_steps` orbital phases as the
    satellites fan out from a narrow polar cluster across the sky."""
    for phase in np.linspace(first_phase, last_phase, n_steps):
        step_emitted = emitted_events(receiver, orbital_planes, phase_offsets + phase, orbit_radius,
                                      angular_speed, n_emission_steps)
        step_observed = step_emitted + time * (c * timestamp_noise)               # [n_satellites] Event-
        step_estimate, _, step_covariance = fitted_reception(step_observed, initial, sigma, n_position_steps)
        yield step_emitted, receiver, step_estimate, position_semi_axes(step_covariance)


recorded_states = show_states(
    sky_spread_states(n_frames),
    render.geometry_animation,
    name="gps_uncertainty",
    duration_ms=frame_duration_ms,
    n_frames=2 * n_frames - 2,
)

With the camera scale held fixed across the orbital sweep, the translucent blue uncertainty ellipsoid starts tall along the polar axis when all six satellites crowd near zenith and contracts sharply as their rays spread toward the horizon.

In [ ]:
# checks
# Four noiseless cones recover the exact reception event, covariance is symmetric, and sky spread shrinks uncertainty.
assert ((cones(located) & located).abs() < mv.scalar(1e-11, unit=dim.length**2)).all()
readouts = stack((mv.x, mv.y, mv.z, mv.t))
assert ((mv.w & (located - receiver)).abs() < 1e-11).all()
assert ((readouts & (located - receiver)[..., None]).abs() < mv.scalar(1e-11, unit=dim.length)).all()
cov_entries = readouts[:, None] & covariance(readouts)
assert ((readouts[:, None] & (covariance - covariance.adjugate())(readouts)).abs() < 1e-12 * cov_entries.abs().max()).all()
narrow_radii = spatial_distance(origin + recorded_states[0][3], origin)            # [3] Scalar+
wide_radii = spatial_distance(origin + recorded_states[-1][3], origin)             # [3] Scalar+
assert (wide_radii < narrow_radii).all()
assert wide_radii[0] * wide_radii[1] * wide_radii[2] < 0.4 * narrow_radii[0] * narrow_radii[1] * narrow_radii[2]